# HR Analytics Project — Data Quality Profiling

## Objective
The goal of this notebook is to inspect the raw HR dataset before cleaning.

This step identifies:
- Row counts
- Missing values
- Duplicate records
- Invalid dates
- Invalid numeric values
- Inconsistent categories
- Data quality issues that need cleaning before SQL analysis

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW_DIR = Path(r"D:\hr_analytics_raw_dataset\hr_analytics_raw_dataset")
PROJECT_DIR = Path(r"D:\hr_analytics_project")
PROFILE_DIR = PROJECT_DIR / "data" / "profile"

PROFILE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR

WindowsPath('D:/hr_analytics_raw_dataset/hr_analytics_raw_dataset')

In [2]:
files = {
    "departments": "raw_departments.csv",
    "job_roles": "raw_job_roles.csv",
    "employees": "raw_employees.csv",
    "salaries": "raw_salaries.csv",
    "performance_reviews": "raw_performance_reviews.csv",
    "satisfaction_surveys": "raw_satisfaction_surveys.csv",
    "training_records": "raw_training_records.csv",
    "attendance": "raw_attendance.csv",
    "attrition_exit_interviews": "raw_attrition_exit_interviews.csv",
    "department_history": "raw_department_history.csv",
}

files

{'departments': 'raw_departments.csv',
 'job_roles': 'raw_job_roles.csv',
 'employees': 'raw_employees.csv',
 'salaries': 'raw_salaries.csv',
 'performance_reviews': 'raw_performance_reviews.csv',
 'satisfaction_surveys': 'raw_satisfaction_surveys.csv',
 'training_records': 'raw_training_records.csv',
 'attendance': 'raw_attendance.csv',
 'attrition_exit_interviews': 'raw_attrition_exit_interviews.csv',
 'department_history': 'raw_department_history.csv'}

In [3]:
raw_data = {}

for table_name, file_name in files.items():
    file_path = RAW_DIR / file_name
    raw_data[table_name] = pd.read_csv(file_path, dtype=str, keep_default_na=False)
    print(f"{table_name}: {raw_data[table_name].shape}")

departments: (14, 6)
job_roles: (35, 8)
employees: (1900, 17)
salaries: (5132, 7)
performance_reviews: (4432, 8)
satisfaction_surveys: (3990, 9)
training_records: (7344, 9)
attendance: (54660, 8)
attrition_exit_interviews: (374, 8)
department_history: (2123, 7)


## 1. Table-Level Profiling

This section checks the size of each table, number of columns, duplicate rows, and blank cells.

In [4]:
def count_blank_values(series):
    return series.astype(str).str.strip().isin(
        ["", "null", "NULL", "None", "none", "nan", "NaN", "N/A", "n/a"]
    ).sum()


table_profile_rows = []

for table_name, df in raw_data.items():
    total_cells = df.shape[0] * df.shape[1]
    total_blank_cells = sum(count_blank_values(df[col]) for col in df.columns)

    table_profile_rows.append({
        "table_name": table_name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "duplicate_full_rows": df.duplicated().sum(),
        "total_blank_cells": total_blank_cells,
        "blank_cell_pct": round(total_blank_cells / total_cells * 100, 2) if total_cells > 0 else 0
    })

table_profile = pd.DataFrame(table_profile_rows)
table_profile

,table_name,row_count,column_count,duplicate_full_rows,total_blank_cells,blank_cell_pct
0,departments,14,6,0,0,0.00
1,job_roles,35,8,0,0,0.00
2,employees,1900,17,20,1162,3.60
3,salaries,5132,7,26,100,0.28
4,performance_reviews,4432,8,0,168,0.47
5,satisfaction_surveys,3990,9,0,539,1.50
6,training_records,7344,9,0,5100,7.72
7,attendance,54660,8,0,53046,12.13
8,attrition_exit_interviews,374,8,0,17,0.57
9,department_history,2123,7,0,1594,10.73


## 2. Column-Level Missing Value Profile

This section checks missing or blank values in each column.

In [5]:
column_profile_rows = []

for table_name, df in raw_data.items():
    for col in df.columns:
        blank_count = count_blank_values(df[col])

        column_profile_rows.append({
            "table_name": table_name,
            "column_name": col,
            "blank_count": blank_count,
            "blank_pct": round(blank_count / len(df) * 100, 2) if len(df) > 0 else 0,
            "unique_values": df[col].nunique(dropna=False),
            "sample_values": " | ".join(df[col].drop_duplicates().astype(str).head(5).tolist())
        })

column_profile = pd.DataFrame(column_profile_rows)
column_profile.sort_values(["blank_pct", "blank_count"], ascending=False).head(30)

,table_name,column_name,blank_count,blank_pct,unique_values,sample_values
70,attendance,absence_type,52719,96.45,8,| UNKNOWN | No Show | Personal Leave | sick
85,department_history,end_date,1521,71.64,556,| 2016-01-05 | 2024/08/07 | 2016-08-25 | Feb ...
60,training_records,completion_date,2732,37.20,2808,2024-02-01 | 27-08-2025 | 2021/04/05 | 2026-03...
62,training_records,training_score,2368,32.24,542,78.5 | 72.8 | 74.1 | 63.1 | 79.1
30,employees,remote_status,321,16.89,6,| Hybrid | Remote | On-site | hybrid
24,employees,manager_id,286,15.05,459,| 100002 | 100003 | 100010 | 100016
27,employees,marital_status,285,15.00,7,Divorced | Single | | Prefer not to say | single
28,employees,education_level,196,10.32,9,Master | Diploma | MBA | | Bachelor
78,attrition_exit_interviews,exit_interview_score,17,4.55,8,5 | 1 | 3 | 4 | 2
86,department_history,change_reason,73,3.44,8,Original Assignment | Lateral Transfer | Initi...


## 3. Date Quality Checks

This section checks whether date columns can be parsed correctly.
Invalid dates will become missing after conversion, so they need cleaning

In [6]:
date_columns = {
    "employees": ["date_of_birth", "hire_date"],
    "salaries": ["effective_date"],
    "performance_reviews": ["review_date"],
    "satisfaction_surveys": ["survey_date"],
    "training_records": ["start_date", "completion_date"],
    "attendance": ["attendance_date"],
    "attrition_exit_interviews": ["exit_date"],
    "department_history": ["start_date", "end_date"],
}


def parse_dates_safely(series):
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


date_quality_rows = []

for table_name, cols in date_columns.items():
    df = raw_data[table_name]

    for col in cols:
        parsed_dates = parse_dates_safely(df[col])
        blank_count = count_blank_values(df[col])
        invalid_count = parsed_dates.isna().sum() - blank_count

        date_quality_rows.append({
            "table_name": table_name,
            "column_name": col,
            "blank_count": blank_count,
            "invalid_date_count": max(int(invalid_count), 0),
            "valid_date_count": int(parsed_dates.notna().sum()),
            "min_date": parsed_dates.min(),
            "max_date": parsed_dates.max()
        })

date_quality = pd.DataFrame(date_quality_rows)
date_quality.sort_values("invalid_date_count", ascending=False)

,table_name,column_name,blank_count,invalid_date_count,valid_date_count,min_date,max_date
7,attendance,attendance_date,62,51,54547,1900-01-01,2026-11-02
5,training_records,start_date,0,21,7323,1900-01-01,2026-12-02
3,performance_reviews,review_date,0,19,4413,1900-01-01,2026-11-01
2,salaries,effective_date,0,17,5115,1900-01-01,2026-12-01
4,satisfaction_surveys,survey_date,0,16,3974,1900-01-01,2026-10-03
6,training_records,completion_date,2732,9,4603,1900-01-01,2026-11-05
1,employees,hire_date,0,5,1895,1900-01-01,2026-12-24
0,employees,date_of_birth,26,3,1871,1900-01-01,2014-08-27
9,department_history,start_date,0,3,2120,2014-01-01,2026-11-01
8,attrition_exit_interviews,exit_date,0,0,374,1900-01-01,2026-11-01


## 4. Numeric Quality Checks

This section checks whether numeric columns contain invalid values, outliers, or blanks.

In [7]:
numeric_columns = {
    "job_roles": ["salary_band_min_usd", "salary_band_max_usd"],
    "salaries": ["salary_amount", "bonus_amount"],
    "performance_reviews": ["performance_score", "manager_score", "goals_met_percentage"],
    "satisfaction_surveys": [
        "satisfaction_score",
        "engagement_score",
        "work_life_balance_score",
        "manager_relationship_score",
        "career_growth_score",
        "compensation_satisfaction_score",
    ],
    "training_records": ["training_score", "training_hours"],
    "attendance": ["scheduled_hours", "worked_hours", "late_minutes"],
}


def parse_numeric_safely(series):
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


numeric_quality_rows = []

for table_name, cols in numeric_columns.items():
    df = raw_data[table_name]

    for col in cols:
        numeric_values = parse_numeric_safely(df[col])
        blank_count = count_blank_values(df[col])
        invalid_count = numeric_values.isna().sum() - blank_count

        numeric_quality_rows.append({
            "table_name": table_name,
            "column_name": col,
            "blank_count": blank_count,
            "invalid_numeric_count": max(int(invalid_count), 0),
            "valid_numeric_count": int(numeric_values.notna().sum()),
            "min_value": numeric_values.min(),
            "max_value": numeric_values.max(),
            "avg_value": numeric_values.mean()
        })

numeric_quality = pd.DataFrame(numeric_quality_rows)
numeric_quality.sort_values("invalid_numeric_count", ascending=False)

,table_name,column_name,blank_count,invalid_numeric_count,valid_numeric_count,min_value,max_value,avg_value
2,salaries,salary_amount,31,71,5030,-362843.2,930243.62,47576.082857
3,salaries,bonus_amount,26,37,5069,-16634.8,112994.39,3734.176658
0,job_roles,salary_band_min_usd,0,0,35,17000.0,115000.00,49485.714286
10,satisfaction_surveys,manager_relationship_score,110,0,3880,1.0,5.00,3.363660
16,attendance,worked_hours,116,0,54544,0.0,21.00,7.800075
15,attendance,scheduled_hours,0,0,54660,7.5,9.00,8.099634
14,training_records,training_hours,0,0,7344,0.8,31.40,10.914583
13,training_records,training_score,2368,0,4976,-5.0,125.00,77.559807
12,satisfaction_surveys,compensation_satisfaction_score,110,0,3880,1.0,5.00,3.129124
11,satisfaction_surveys,career_growth_score,103,0,3887,1.0,5.00,3.275791


## 5. Categorical Value Checks

This section checks inconsistent text values such as department names, job titles, employee status, and training status.

In [8]:
category_columns = {
    "employees": [
        "gender",
        "department_name_raw",
        "job_title_raw",
        "employment_type",
        "location",
        "status_raw",
        "remote_status",
    ],
    "departments": ["department_name", "business_unit", "region", "active_flag"],
    "job_roles": ["job_title", "canonical_title", "job_level", "role_family"],
    "training_records": ["completion_status", "training_category"],
    "attendance": ["absence_flag", "absence_type"],
    "attrition_exit_interviews": ["exit_type", "exit_reason", "rehire_eligible"],
}

category_value_rows = []

for table_name, cols in category_columns.items():
    df = raw_data[table_name]

    for col in cols:
        value_counts = (
            df[col]
            .astype(str)
            .str.strip()
            .value_counts(dropna=False)
            .head(25)
        )

        for value, count in value_counts.items():
            category_value_rows.append({
                "table_name": table_name,
                "column_name": col,
                "value": value,
                "count": count
            })

category_values = pd.DataFrame(category_value_rows)
category_values.head(50)

,table_name,column_name,value,count
0,employees,gender,Male,862
1,employees,gender,Female,827
2,employees,gender,Non-binary,44
3,employees,gender,Man,27
4,employees,gender,male,26
5,employees,gender,Prefer not to say,25
6,employees,gender,female,24
7,employees,gender,F,23
8,employees,gender,M,22
9,employees,gender,Woman,18


In [9]:
category_values[
    (category_values["table_name"] == "employees") &
    (category_values["column_name"] == "department_name_raw")
].sort_values("count", ascending=False)

,table_name,column_name,value,count
12,employees,department_name_raw,Cust Support,127
13,employees,department_name_raw,Customer support,106
14,employees,department_name_raw,CUSTOMER SUPPORT,101
15,employees,department_name_raw,Support,101
16,employees,department_name_raw,Customer Support,100
17,employees,department_name_raw,Engineering,76
18,employees,department_name_raw,Eng,74
19,employees,department_name_raw,Software Engineering,72
20,employees,department_name_raw,engineering,66
21,employees,department_name_raw,ENGINEERING,64


In [10]:
category_values[
    (category_values["table_name"] == "employees") &
    (category_values["column_name"] == "job_title_raw")
].sort_values("count", ascending=False)

,table_name,column_name,value,count
37,employees,job_title_raw,customer support associate,101
38,employees,job_title_raw,Support Associate,94
39,employees,job_title_raw,Cust Support Agent,92
40,employees,job_title_raw,Customer Support Associate,81
41,employees,job_title_raw,Operations Analyst,45
42,employees,job_title_raw,Acct Executive,41
43,employees,job_title_raw,Engineering Mgr,38
44,employees,job_title_raw,Ops Analyst,37
45,employees,job_title_raw,Account Executive,37
46,employees,job_title_raw,SDR,37


In [11]:
category_values[
    (category_values["table_name"] == "employees") &
    (category_values["column_name"] == "status_raw")
].sort_values("count", ascending=False)

,table_name,column_name,value,count
75,employees,status_raw,active,342
76,employees,status_raw,Still Working,313
77,employees,status_raw,Active,310
78,employees,status_raw,ACTIVE,300
79,employees,status_raw,Current,269
80,employees,status_raw,Resigned,69
81,employees,status_raw,left,63
82,employees,status_raw,Exited,60
83,employees,status_raw,Left,59
84,employees,status_raw,Inactive,57


## 6. Export Profiling Reports

The profiling reports are exported as CSV files so they can be included in the project folder and referenced in the README.

In [12]:
table_profile.to_csv(PROFILE_DIR / "table_profile.csv", index=False)
column_profile.to_csv(PROFILE_DIR / "column_profile.csv", index=False)
date_quality.to_csv(PROFILE_DIR / "date_quality.csv", index=False)
numeric_quality.to_csv(PROFILE_DIR / "numeric_quality.csv", index=False)
category_values.to_csv(PROFILE_DIR / "category_values.csv", index=False)

print(f"Profiling reports exported to: {PROFILE_DIR}")

Profiling reports exported to: D:\hr_analytics_project\data\profile
